In [ ]:
# =========================================================
# collect_results.py
# Run this anytime after submit_batches.py — even in a
# completely new Kaggle session, hours or days later.
# It pulls results for any batches that have finished,
# and is safe to rerun repeatedly (skips already-collected ones).
# =========================================================
import os
import json
import pandas as pd
from datasets import load_dataset
from openai import OpenAI
from huggingface_hub import HfApi, hf_hub_download, login

# =========================================================
# Config
# =========================================================
os.environ.setdefault("OPENAI_API_KEY", "sk_key") # Access key removed for safety
os.environ["HF_TOKEN"] = "hf_token" # Acess token removed for safety
login(os.environ["HF_TOKEN"])

TEST_DATASET_NAME = "businessrules/dataset_stratified_test"
TRACKING_REPO = "businessrules/GPT4_batch_tracking"
TRACKING_FILE = "batch_jobs.json"
OUTPUT_DATASET_NAME = "businessrules/GPT4_baseline_results"
RESULTS_FILE = "results.json"  
client = OpenAI()
api = HfApi()


# =========================================================
# Load tracking (which batches were submitted) and any
# results already collected in a prior run of this script.
# =========================================================
def load_json_from_hf(repo_id, filename, default):
    try:
        path = hf_hub_download(repo_id=repo_id, repo_type="dataset", filename=filename)
        with open(path, "r") as f:
            return json.load(f)
    except Exception:
        return default


def save_json_to_hf(obj, repo_id, filename, local_path):
    with open(local_path, "w") as f:
        json.dump(obj, f, indent=2)
    api.create_repo(repo_id=repo_id, repo_type="dataset", exist_ok=True)
    api.upload_file(
        path_or_fileobj=local_path,
        path_in_repo=filename,
        repo_id=repo_id,
        repo_type="dataset",
    )


tracking = load_json_from_hf(TRACKING_REPO, TRACKING_FILE, {"submitted_batches": [], "submitted_ids": []})
results = load_json_from_hf(OUTPUT_DATASET_NAME, RESULTS_FILE, [])
collected_ids = {r["id"] for r in results}

test_dataset = load_dataset(TEST_DATASET_NAME, split="test")

# --- Must match the SLICE_RANGE used in submit_batches.py ---
SLICE_RANGE = None
if SLICE_RANGE is not None:
    test_dataset = test_dataset.select(SLICE_RANGE)

example_by_id = {str(ex["id"]): ex for ex in test_dataset}

pending = [b for b in tracking["submitted_batches"] if not b["collected"]]
print(f"{len(pending)} batch(es) not yet collected. Checking status...")

any_updates = False

for batch_entry in pending:
    batch_id = batch_entry["batch_id"]
    batch_job = client.batches.retrieve(batch_id)
    status = batch_job.status
    print(f"Batch {batch_id}: status = {status}")

    if status != "completed":

        if status in ("failed", "expired", "cancelled"):
            print(f"  Batch {batch_id} ended as '{status}' — some requests may need resubmitting.")
        continue

    output_file_id = batch_job.output_file_id
    file_response = client.files.content(output_file_id)

    for line in file_response.text.splitlines():
        record = json.loads(line)
        custom_id = record["custom_id"]
        if custom_id in collected_ids:
            continue  

        body = record["response"]["body"]
        prediction = body["choices"][0]["message"]["content"].strip()
        example = example_by_id[custom_id]

        results.append({
            "id": example["id"],
            "golden_business_rule": example["br"],
            "gpt4_prediction": prediction,
            "gpt4_model": body.get("model", "gpt-4o"),
        })
        collected_ids.add(custom_id)

    batch_entry["collected"] = True
    any_updates = True

    # Save progress to HF immediately after EACH batch — not at the end.
    save_json_to_hf(results, OUTPUT_DATASET_NAME, RESULTS_FILE, RESULTS_FILE)
    save_json_to_hf(tracking, TRACKING_REPO, TRACKING_FILE, TRACKING_FILE)
    print(f"  Collected batch {batch_id}. Total results so far: {len(results)}/{len(test_dataset)}")

if not any_updates:
    print("\nNo newly completed batches this run. Try again later.")

# =========================================================
# Also publish as a clean parquet-backed dataset for easy reading
# =========================================================
if results:
    df = pd.DataFrame(results)
    df.to_parquet("results.parquet")
    api.upload_file(
        path_or_fileobj="results.parquet",
        path_in_repo="data.parquet",
        repo_id=OUTPUT_DATASET_NAME,
        repo_type="dataset",
    )
    print(f"\n{len(results)}/{len(test_dataset)} total results available at "
          f"https://huggingface.co/datasets/{OUTPUT_DATASET_NAME}")

    still_pending = [b for b in tracking["submitted_batches"] if not b["collected"]]
    if still_pending:
        print(f" {len(still_pending)} batch(es) still pending — rerun this script later.")